# Move Fast, Don't Break Things (with dbt)

- A tiny dbt project built on NYC Open Data: the 2024 Civil List and Civil Service Titles
- **Part 1:** tour the project
- **Part 2:** how the CI pipeline works
- **Part 3:** make a "simple change" and let CI catch our mistakes

In [2]:
import os
import re
import subprocess
import sysconfig
from pathlib import Path

import duckdb

# Make `!dbt` use the dbt installed in this notebook's environment
os.environ["PATH"] = sysconfig.get_path("scripts") + os.pathsep + os.environ["PATH"]
print(subprocess.run(["dbt", "--version"], check=True, capture_output=True).stdout.decode())  # fails loudly if dbt is missing


def query(sql, db="data/dev.duckdb"):
    # Read-only mode not supported for in-memory databases
    read_only = True if db != ":memory:" else False
    with duckdb.connect(db, read_only=read_only) as con:
        con.sql(sql).show(max_width=250)

def show_file(path):
    print(Path(path).read_text())

def use_branch(branch):
    """Swap in the models/ folder from a demo branch (this notebook stays put), and show how it differs from main."""
    subprocess.run(["git", "restore", f"--source={branch}", "--", "models"], check=True)
    diff = subprocess.run(
        ["git", "--no-pager", "diff", "--color=always", "main", "--", "models"],
        capture_output=True, text=True, check=True,
    )
    print(diff.stdout)

Core:
  - installed: 1.12.6
  - latest:    1.12.6 - Up to date!

Plugins:
  - duckdb: 1.11.0 - Up to date!


13:51:02  dbt v2 is available, and is the default for new installations. This is dbt Core v1.
          New project - install v2: https://docs.getdbt.com/docs/local/install-dbt?utm_source=dbt-cli
          Existing v1 project - upgrade guide: https://docs.getdbt.com/docs/dbt-versions/dbt-upgrade/upgrading-to-v2?utm_source=dbt-cli



## 1. Project overview

- `data/snapshots/`: Parquet snapshots of two Open Data datasets. These are dbt **sources**: read-only inputs
- `models/staging/`: light cleanup of each source (rename columns, fix types)
- `models/legacy/`: the "giant evil SQL" report behind the Commissioner's Dashboard
- `profiles.yml`: where dbt builds things. Here, that's DuckDB files in `data/`

### The data we're working with....

* The [Civil Service Titles dataset](https://data.cityofnewyork.us/City-Government/NYC-Civil-Service-Titles/nzjr-3966/about_data) contains one row for every civil service title code.
* Each assignment level within a title has its own code.

In [3]:

query("select count(*) from 'data/snapshots/civil_service_titles.parquet'", db=":memory:")
query("select * from 'data/snapshots/civil_service_titles.parquet' limit 5", db=":memory:")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│         3385 │
└──────────────┘

┌────────────────────┬───────────────────┬───────────────────────┬──────────────────────┬─────────┬──────────────────────┬─────────┬───┬──────────┬──────────────────────┬───────────┬──────────────────────┬──────────┬──────────┬──────────────────────┐
│         id         │     _version      │      created_at       │      updated_at      │  title  │        descr         │ std_hrs │ … │ union_cd │     union_descr      │ barg_unit │      barg_descr      │ min_rate │ max_rate │ investigation_befor… │
│      varchar       │      varchar      │ timestamp with time … │ timestamp with time… │ varchar │       varchar        │ double  │ … │ varchar  │       varchar        │  varchar  │       varchar        │  double  │  double  │       boolean        │
├────────────────────┼───────────────────┼───────────────────────┼──────────────────────┼─────────┼──────────────────────┼─────────┼───┼────────

* The [Civil List dataset](https://data.cityofnewyork.us/City-Government/Civil-List/ye3c-m4ga/about_data) contains one row for each person employed by the City at any point during a given year.
* This extract only contains data from 2024.

In [4]:
query("select count(*) from 'data/snapshots/civil_list.parquet'", db=":memory:")
query("select * from 'data/snapshots/civil_list.parquet' limit 5", db=":memory:")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       289692 │
└──────────────┘

┌───────────────┬───────┬────────────────────────────┬────────────────┬─────────┬─────────┬────────────┬────────────────────┬───────────────────┬────────────────────────────┬────────────────────────────┐
│ calendar_year │  dpt  │        agency_name         │     _name      │   ttl   │   pc    │  sal_rate  │         id         │     _version      │         created_at         │         updated_at         │
│     int64     │ int64 │          varchar           │    varchar     │ varchar │ varchar │  varchar   │      varchar       │      varchar      │  timestamp with time zone  │  timestamp with time zone  │
├───────────────┼───────┼────────────────────────────┼────────────────┼─────────┼─────────┼────────────┼────────────────────┼───────────────────┼────────────────────────────┼────────────────────────────┤
│          2024 │   998 │ N.Y.C. TRANSIT AUTHORITY   │ S J A          │ 00581   │

### `dbt debug`

- Checks that dbt can find the project and the profile, and connect to the database

In [5]:
!dbt debug

13:51:03  Running with dbt=1.12.6
13:51:03  dbt version: 1.12.6
13:51:03  python version: 3.12.3
13:51:03  python path: /home/gstrauss/glqstrauss/anex-2026-demo/.venv/bin/python3
13:51:03  os info: Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.39
13:51:04  Using profiles dir at /home/gstrauss/glqstrauss/anex-2026-demo
13:51:04  Using profiles.yml file at /home/gstrauss/glqstrauss/anex-2026-demo/profiles.yml
13:51:04  Using dbt_project.yml file at /home/gstrauss/glqstrauss/anex-2026-demo/dbt_project.yml
13:51:04  adapter type: duckdb
13:51:04  adapter version: 1.11.0
13:51:04  Configuration:
13:51:04    profiles.yml file [OK found and valid]
13:51:04    dbt_project.yml file [OK found and valid]
13:51:04  Required dependencies:
13:51:04   - git [OK found]

13:51:04  Connection:
13:51:04    database: dev
13:51:04    schema: main
13:51:04    path: data/dev.duckdb
13:51:04    config_options: None
13:51:04    extensions: None
13:51:04    settings: {}
13:51:04    external_root: .


### `dbt build`

- Runs every model, in dependency order, as a table in the **dev** database
- Runs each model's tests right after it's built
- Prod and CI run this same command; only the `--target` changes

In [6]:
!dbt build

13:51:06  Running with dbt=1.12.6
13:51:06  Registered adapter: duckdb=1.11.0
13:51:06  Found 3 models, 4 data tests, 3 sources, 1 exposure, 500 macros
13:51:06  
13:51:06  Concurrency: 1 threads (target='dev')
13:51:06  
13:51:07  1 of 8 START sql table model main_civil_service.stg_civil_servants ............. [RUN]
13:51:07  1 of 8 OK created sql table model main_civil_service.stg_civil_servants ........ [OK in 0.33s]
13:51:07  2 of 8 START sql table model main_civil_service.stg_civil_service_titles ....... [RUN]
13:51:07  2 of 8 OK created sql table model main_civil_service.stg_civil_service_titles .. [OK in 0.05s]
13:51:07  3 of 8 START sql table model main_civil_service.rpt_agency_title_payroll ....... [RUN]
13:51:07  3 of 8 OK created sql table model main_civil_service.rpt_agency_title_payroll .. [OK in 0.13s]
13:51:07  4 of 8 NO-OP exposure commissioners_dashboard .................................. [NO-OP in 0.00s]
13:51:07  5 of 8 START test accepted_values_rpt_agency_title_pay

### What did that create?

- DuckDB keeps a whole database in a single file: `data/dev.duckdb`
- Each model is now a table in the `main_civil_service` schema

In [7]:
query("show;")

┌──────────┬────────────────────┬──────────────────────────┬────────────────────────────────────────────────────────────────────────────────────────┬────────────────────────────────────────────────────────────────────────────────────────┬───────────┐
│ database │       schema       │           name           │                                      column_names                                      │                                      column_types                                      │ temporary │
│ varchar  │      varchar       │         varchar          │                                       varchar[]                                        │                                       varchar[]                                        │  boolean  │
├──────────┼────────────────────┼──────────────────────────┼────────────────────────────────────────────────────────────────────────────────────────┼────────────────────────────────────────────────────────────────────────────────────────┼─────────

In [8]:
query("from main_civil_service.rpt_agency_title_payroll order by headcount desc limit 10")

┌─────────────┬────────────────────────────┬──────────────┬────────────┬────────────────────────────────┬───────────┬─────────────────────┬───────────────┐
│ agency_code │        agency_name         │ agency_group │ title_code │           title_desc           │ headcount │ total_annual_salary │  salary_band  │
│   varchar   │          varchar           │   varchar    │  varchar   │            varchar             │   int64   │    decimal(14,2)    │    varchar    │
├─────────────┼────────────────────────────┼──────────────┼────────────┼────────────────────────────────┼───────────┼─────────────────────┼───────────────┤
│ 056         │ POLICE DEPARTMENT          │ UNIFORMED    │ 70210      │ POLICE OFFICER (RECURRING NIGH │     22742 │       1960482333.00 │ 3 - 70K-100K  │
│ 868         │ DEPARTMENT OF CITYWIDE ADM │ OTHER        │ 12702      │ MONITOR                        │     11442 │        376958400.00 │ 1 - UNDER 40K │
│ 057         │ FIRE DEPARTMENT            │ UNIFORMED    │ 7031

### The legacy report

- Written years ago and documented by no one
- Or maybe written by you a year ago. It's hard to remember the details!
- Uppercase everything, three levels of nested subqueries, magic numbers, commented-out code
- "DO NOT CHANGE w/o talking to RK". Nobody knows who RK is

In [9]:
show_file("models/legacy/rpt_agency_title_payroll.sql")

-- AGENCY TITLE PAYROLL REPORT
-- feeds commissioner dashboard. DO NOT CHANGE w/o talking to RK
-- last updated 3/2019
SELECT
	X.AGENCY_CODE,
	X.AGENCY_NAME,
	X.AGENCY_GROUP,
	X.TITLE_CODE,
	X.TITLE_DESC,
	X.HEADCOUNT,
	X.TOTAL_ANNUAL_SALARY,
	CASE
		WHEN X.TOTAL_ANNUAL_SALARY / X.HEADCOUNT < 40000 THEN '1 - UNDER 40K'
		WHEN X.TOTAL_ANNUAL_SALARY / X.HEADCOUNT < 70000 THEN '2 - 40K-70K'
		WHEN X.TOTAL_ANNUAL_SALARY / X.HEADCOUNT <= 100000 THEN '3 - 70K-100K'
		WHEN X.TOTAL_ANNUAL_SALARY / X.HEADCOUNT >= 100000 THEN '4 - 100K+'
	END AS salary_band
FROM (
	SELECT
		B.AGENCY_CODE,
		B.AGENCY_NAME,
		B.AGENCY_GROUP,
		B.TITLE_CODE,
		B.TITLE_DESC,
		COUNT(*) AS headcount,
		ROUND(SUM(B.ANNUAL_SALARY), 2)::DECIMAL(14, 2) AS total_annual_salary
	FROM (
		SELECT
			A.*,
			-- fix truncated names for dashboard
			CASE
				WHEN A.AGENCY_NAME = 'DEPARTMENT OF CITYWIDE ADM' THEN 'DEPARTMENT OF CITYWIDE ADMINISTRATIVE SERVICES'
				WHEN A.AGENCY_NAME = 'DEPARTMENT OF PARKS & RECR' THEN 'DEPARTMEN

## 2. How CI works

- **Goal:** check every pull request automatically, before it can affect anything real
- **Prod state:** every merge to `main` runs `deploy.yml`, which builds prod and saves its `manifest.json`, dbt's complete description of the project
- **Isolation:** CI builds into its own schema (`ci_pr_<number>`), so it never touches prod tables
- **State comparison:** CI compares the pull request's manifest with prod's to work out what changed

```
merge to main ──▶ deploy.yml: build prod ──▶ prod database + manifest.json
                                                     │
open a PR ────▶ ci.yml: download prod state ──▶ build what changed
```

### The CI build

| Flag | What it does |
| --- | --- |
| `--target ci` | Build into the isolated CI schema |
| `--select state:modified+` | Build models that changed plus everything downstream |
| `--defer --favor-state` | For everything else, read prod's tables instead of rebuilding them |
| `--state data/prod_state` | Where to find prod's manifest |
| `--empty` | A first pass with zero rows: checks the SQL and contracts in seconds |

- CI runs `dbt build` twice: first with `--empty`, then for real

In [10]:
show_file(".github/workflows/ci.yml")

name: CI

on:
  pull_request:

jobs:
  ci:
    runs-on: ubuntu-latest
    env:
      CI_SCHEMA_SUFFIX: pr_${{ github.event.pull_request.number }}
    steps:
      - uses: actions/checkout@v5

      - uses: actions/setup-python@v6
        with:
          python-version: "3.12"
          cache: pip

      - name: Install dependencies
        run: pip install -r requirements.txt

      - name: Download prod state
        env:
          GH_TOKEN: ${{ github.token }}
        run: |
          mkdir -p data/prod_state
          gh release download prod-state --pattern prod.duckdb --dir data
          gh release download prod-state --pattern manifest.json --dir data/prod_state

      # Build only what changed (state:modified+) into the CI schema, and read
      # everything else from prod (--defer --favor-state)
      - name: Check SQL and contracts (no data)
        run: dbt build --target ci --select state:modified+ --defer --favor-state --state data/prod_state --empty

      - name: Build a

In [11]:
ci_flags = "--target ci --select state:modified+ --defer --favor-state --state data/prod_state"

## 3. Let's make a simple change

- **The ask:** Labor Relations needs union info added to the payroll report.
- Live demos can be harmful to your health! Instead, we'll swap in pre-configured branches and run CI on them...
- First, build prod from `main`. On GitHub, this happens on every merge

In [12]:
!git restore models
!dbt build --target prod --target-path data/prod_state

13:51:10  Running with dbt=1.12.6
13:51:10  Registered adapter: duckdb=1.11.0
13:51:10  Found 3 models, 4 data tests, 3 sources, 1 exposure, 500 macros
13:51:10  
13:51:10  Concurrency: 4 threads (target='prod')
13:51:10  
13:51:11  1 of 8 START sql table model main_civil_service.stg_civil_servants ............. [RUN]
13:51:11  2 of 8 START sql table model main_civil_service.stg_civil_service_titles ....... [RUN]
13:51:11  2 of 8 OK created sql table model main_civil_service.stg_civil_service_titles .. [OK in 0.15s]
13:51:11  1 of 8 OK created sql table model main_civil_service.stg_civil_servants ........ [OK in 0.33s]
13:51:11  3 of 8 START sql table model main_civil_service.rpt_agency_title_payroll ....... [RUN]
13:51:11  3 of 8 OK created sql table model main_civil_service.rpt_agency_title_payroll .. [OK in 0.14s]
13:51:11  5 of 8 START test accepted_values_rpt_agency_title_payroll_agency_group__UNIFORMED__EDUCATION__HEALTH_HUMAN_SVCS__DISTRICT_ATTORNEYS__OTHER  [RUN]
13:51:11  6 of

### Step 1: A harmless cleanup

- `salary_rate` is hourly or daily for many employees, so we should rename it to `pay_rate`!
- Not the task at hand but might as well be a good citizen....
- We'll search/replace "salary_rate" in the code....

In [13]:
use_branch("origin/demo/1-rename")

diff --git a/models/staging/stg_civil_servants.sql b/models/staging/stg_civil_servants.sql
index ff2e8cf..f1fd370 100644
--- a/models/staging/stg_civil_servants.sql
+++ b/models/staging/stg_civil_servants.sql
@@ -5,4 +5,4 @@ select
 	_name as employee_name,
 	ttl as title_code,
 	pc as pay_class,
-	regexp_replace(sal_rate, '[$,]','', 'g')::double as salary_rate
+	regexp_replace(sal_rate, '[$,]','', 'g')::double as pay_rate



In [14]:
!dbt build {ci_flags} --empty

13:51:13  Running with dbt=1.12.6
13:51:14  Registered adapter: duckdb=1.11.0
13:51:14  Unable to do partial parsing because config vars, config profile, or config target have changed
13:51:14  Unable to do partial parsing because profile has changed
13:51:16  Found 3 models, 4 data tests, 3 sources, 1 exposure, 500 macros
13:51:16  
13:51:16  Concurrency: 4 threads (target='ci')
13:51:16  
13:51:16  1 of 7 START sql table model ci_local_civil_service.stg_civil_servants ......... [RUN]
13:51:16  1 of 7 OK created sql table model ci_local_civil_service.stg_civil_servants .... [OK in 0.13s]
13:51:16  2 of 7 START sql table model ci_local_civil_service.rpt_agency_title_payroll ... [RUN]
13:51:16  2 of 7 ERROR creating sql table model ci_local_civil_service.rpt_agency_title_payroll  [ERROR in 0.04s]
13:51:16  3 of 7 SKIP exposure commissioners_dashboard ................................... [SKIP]
13:51:16  4 of 7 SKIP test accepted_values_rpt_agency_title_payroll_agency_group__UNIFORMED__ED

* The payroll report is all uppercase, so our search/replace missed it!
* We caught this error without touching data (query engine dependent....)

### Step 2: Add the union, first try

- Add `UNION_DESC` to the report

In [15]:
use_branch("origin/demo/2-contract")

diff --git a/models/legacy/rpt_agency_title_payroll.sql b/models/legacy/rpt_agency_title_payroll.sql
index 3a9061b..48604ab 100644
--- a/models/legacy/rpt_agency_title_payroll.sql
+++ b/models/legacy/rpt_agency_title_payroll.sql
@@ -7,6 +7,7 @@ SELECT
 	X.AGENCY_GROUP,
 	X.TITLE_CODE,
 	X.TITLE_DESC,
+	X.UNION_DESC,
 	X.HEADCOUNT,
 	X.TOTAL_ANNUAL_SALARY,
 	CASE
@@ -22,6 +23,7 @@ FROM (
 		B.AGENCY_GROUP,
 		B.TITLE_CODE,
 		B.TITLE_DESC,
+		B.UNION_DESC,
 		COUNT(*) AS headcount,
 		ROUND(SUM(B.ANNUAL_SALARY), 2)::DECIMAL(14, 2) AS total_annual_salary
 	FROM (
@@ -56,13 +58,14 @@ FROM (
 			SELECT
 				E.*,
 				T.TITLE_DESC,
+				T.UNION_DESC,
 				(SELECT MAX(T2.STANDARD_HOURS) FROM {{ ref("stg_civil_service_titles") }} T2 WHERE T2.TITLE_CODE = E.TITLE_CODE) AS std_hrs
 			FROM {{ ref("stg_civil_servants") }} E
-			JOIN (SELECT DISTINCT TITLE_CODE, TITLE_DESC FROM {{ ref("stg_civil_service_titles") }}) T
+			JOIN (SELECT DISTINCT TITLE_CODE, TITLE_DESC, UNION_DESC FROM {{ ref("stg_c

In [16]:
!dbt build {ci_flags} --empty

13:51:18  Running with dbt=1.12.6
13:51:18  Registered adapter: duckdb=1.11.0
13:51:19  Found 3 models, 4 data tests, 3 sources, 1 exposure, 500 macros
13:51:19  
13:51:19  Concurrency: 4 threads (target='ci')
13:51:19  
13:51:20  1 of 6 START sql table model ci_local_civil_service.rpt_agency_title_payroll ... [RUN]
13:51:20  1 of 6 ERROR creating sql table model ci_local_civil_service.rpt_agency_title_payroll  [ERROR in 0.11s]
13:51:20  2 of 6 SKIP exposure commissioners_dashboard ................................... [SKIP]
13:51:20  3 of 6 SKIP test accepted_values_rpt_agency_title_payroll_agency_group__UNIFORMED__EDUCATION__HEALTH_HUMAN_SVCS__DISTRICT_ATTORNEYS__OTHER  [SKIP]
13:51:20  4 of 6 SKIP test not_null_rpt_agency_title_payroll_agency_code ................. [SKIP]
13:51:20  5 of 6 SKIP test not_null_rpt_agency_title_payroll_title_code .................. [SKIP]
13:51:20  6 of 6 SKIP test unique_rpt_agency_title_payroll_agency_code_title_code ........ [SKIP]
13:51:20  
13:51:20

- The report has an enforced **contract**: its columns and types are declared in `_legacy.yml`
- A column the contract doesn't list fails the build before any data is read
- Behind the scenes, dbt is replacing CREATE TABLE AS SELECT with CREATE TABLE followed by INSERT. The database is doing the contract enforcement...

### Step 3: Update the contract

- Add `union_desc` to the contract and try again

In [17]:
use_branch("origin/demo/3-fanout")

diff --git a/models/legacy/_legacy.yml b/models/legacy/_legacy.yml
index bebe726..3f90815 100644
--- a/models/legacy/_legacy.yml
+++ b/models/legacy/_legacy.yml
@@ -23,6 +23,8 @@ models:
           - not_null
       - name: title_desc
         data_type: varchar
+      - name: union_desc
+        data_type: varchar
       - name: headcount
         data_type: bigint
       - name: total_annual_salary
diff --git a/models/legacy/rpt_agency_title_payroll.sql b/models/legacy/rpt_agency_title_payroll.sql
index 3a9061b..48604ab 100644
--- a/models/legacy/rpt_agency_title_payroll.sql
+++ b/models/legacy/rpt_agency_title_payroll.sql
@@ -7,6 +7,7 @@ SELECT
 	X.AGENCY_GROUP,
 	X.TITLE_CODE,
 	X.TITLE_DESC,
+	X.UNION_DESC,
 	X.HEADCOUNT,
 	X.TOTAL_ANNUAL_SALARY,
 	CASE
@@ -22,6 +23,7 @@ FROM (
 		B.AGENCY_GROUP,
 		B.TITLE_CODE,
 		B.TITLE_DESC,
+		B.UNION_DESC,
 		COUNT(*) AS headcount,
 		ROUND(SUM(B.ANNUAL_SALARY), 2)::DECIMAL(14, 2) AS total_annual_salary
 	FROM (
@@ -56,13 +58,14 @@ FROM (
 

In [18]:
!dbt build {ci_flags} --empty

13:51:22  Running with dbt=1.12.6
13:51:22  Registered adapter: duckdb=1.11.0
13:51:23  Found 3 models, 4 data tests, 3 sources, 1 exposure, 500 macros
13:51:23  
13:51:23  Concurrency: 4 threads (target='ci')
13:51:23  
13:51:23  1 of 6 START sql table model ci_local_civil_service.rpt_agency_title_payroll ... [RUN]
13:51:23  1 of 6 OK created sql table model ci_local_civil_service.rpt_agency_title_payroll  [OK in 0.17s]
13:51:23  3 of 6 START test accepted_values_rpt_agency_title_payroll_agency_group__UNIFORMED__EDUCATION__HEALTH_HUMAN_SVCS__DISTRICT_ATTORNEYS__OTHER  [RUN]
13:51:23  4 of 6 START test not_null_rpt_agency_title_payroll_agency_code ................ [RUN]
13:51:23  5 of 6 START test not_null_rpt_agency_title_payroll_title_code ................. [RUN]
13:51:23  2 of 6 NO-OP exposure commissioners_dashboard .................................. [NO-OP in 0.00s]
13:51:23  6 of 6 START test unique_rpt_agency_title_payroll_agency_code_title_code ....... [RUN]
13:51:23  5 of 6 PA

- `--empty` passed: the SQL and the contract are fine. On to the real build:

In [19]:
!dbt build {ci_flags}

13:51:26  Running with dbt=1.12.6
13:51:26  Registered adapter: duckdb=1.11.0
13:51:26  Found 3 models, 4 data tests, 3 sources, 1 exposure, 500 macros
13:51:26  
13:51:26  Concurrency: 4 threads (target='ci')
13:51:26  
13:51:26  1 of 6 START sql table model ci_local_civil_service.rpt_agency_title_payroll ... [RUN]
13:51:27  1 of 6 OK created sql table model ci_local_civil_service.rpt_agency_title_payroll  [OK in 0.22s]
13:51:27  3 of 6 START test accepted_values_rpt_agency_title_payroll_agency_group__UNIFORMED__EDUCATION__HEALTH_HUMAN_SVCS__DISTRICT_ATTORNEYS__OTHER  [RUN]
13:51:27  4 of 6 START test not_null_rpt_agency_title_payroll_agency_code ................ [RUN]
13:51:27  5 of 6 START test not_null_rpt_agency_title_payroll_title_code ................. [RUN]
13:51:27  2 of 6 NO-OP exposure commissioners_dashboard .................................. [NO-OP in 0.00s]
13:51:27  6 of 6 START test unique_rpt_agency_title_payroll_agency_code_title_code ....... [RUN]
13:51:27  5 of 6 PA

- The `unique` test failed: 422 agency/title rows are duplicated.
- Some titles belong to more than one union, so the `DISTINCT` subquery now returns several rows per title, and their headcounts are inflated.

In [20]:
query("""
select agency_name, title_code, title_desc, union_desc 
from ci_local_civil_service.rpt_agency_title_payroll
qualify count(*) over(partition by agency_code, title_code) > 1
order by agency_code, title_code
limit 20
""", db="data/prod.duckdb")

┌────────────────────────────┬────────────┬───────────────────────────────┬────────────────────────────────┐
│        agency_name         │ title_code │          title_desc           │           union_desc           │
│          varchar           │  varchar   │            varchar            │            varchar             │
├────────────────────────────┼────────────┼───────────────────────────────┼────────────────────────────────┤
│ OFFICE OF THE ACTUARY      │ 56057      │ COMMUNITY ASSOCIATE           │ NEIGHBORHOOD WORKER BD ED DC37 │
│ OFFICE OF THE ACTUARY      │ 56057      │ COMMUNITY ASSOCIATE           │ SOCIAL SERVICE TITLES DC 37L37 │
│ NYC EMPLOYEES' RETIREMENT  │ 54736      │ CONFIDENTIAL STRATEGY PLANNER │ STAFF ANALYSTS                 │
│ NYC EMPLOYEES' RETIREMENT  │ 54736      │ CONFIDENTIAL STRATEGY PLANNER │ NON COMPETITIVE OJ             │
│ NYC EMPLOYEES' RETIREMENT  │ 56056      │ COMMUNITY ASSISTANT           │ NEIGHBORHOOD WORKER BD ED DC37 │
│ NYC EMPLOYEES' RE

### Step 4: The right way

- We go to the SME and ask them how they want the data represented.
- Solution: make the title subquery one row per title, combining multiple unions into one value.


In [21]:
use_branch("origin/demo/4-right-way")

diff --git a/models/legacy/_legacy.yml b/models/legacy/_legacy.yml
index bebe726..ce18ef5 100644
--- a/models/legacy/_legacy.yml
+++ b/models/legacy/_legacy.yml
@@ -23,6 +23,10 @@ models:
           - not_null
       - name: title_desc
         data_type: varchar
+      - name: union_desc
+        data_type: varchar
+        data_tests:
+          - not_null
       - name: headcount
         data_type: bigint
       - name: total_annual_salary
diff --git a/models/legacy/rpt_agency_title_payroll.sql b/models/legacy/rpt_agency_title_payroll.sql
index 3a9061b..ebcc73e 100644
--- a/models/legacy/rpt_agency_title_payroll.sql
+++ b/models/legacy/rpt_agency_title_payroll.sql
@@ -7,6 +7,7 @@ SELECT
 	X.AGENCY_GROUP,
 	X.TITLE_CODE,
 	X.TITLE_DESC,
+	X.UNION_DESC,
 	X.HEADCOUNT,
 	X.TOTAL_ANNUAL_SALARY,
 	CASE
@@ -22,6 +23,7 @@ FROM (
 		B.AGENCY_GROUP,
 		B.TITLE_CODE,
 		B.TITLE_DESC,
+		B.UNION_DESC,
 		COUNT(*) AS headcount,
 		ROUND(SUM(B.ANNUAL_SALARY), 2)::DECIMAL(14, 2) AS total_annual_

In [22]:
!dbt build {ci_flags} --empty

13:51:29  Running with dbt=1.12.6
13:51:29  Registered adapter: duckdb=1.11.0
13:51:30  Found 3 models, 5 data tests, 3 sources, 1 exposure, 500 macros
13:51:30  
13:51:30  Concurrency: 4 threads (target='ci')
13:51:30  
13:51:30  1 of 7 START sql table model ci_local_civil_service.rpt_agency_title_payroll ... [RUN]
13:51:31  1 of 7 OK created sql table model ci_local_civil_service.rpt_agency_title_payroll  [OK in 0.17s]
13:51:31  3 of 7 START test accepted_values_rpt_agency_title_payroll_agency_group__UNIFORMED__EDUCATION__HEALTH_HUMAN_SVCS__DISTRICT_ATTORNEYS__OTHER  [RUN]
13:51:31  4 of 7 START test not_null_rpt_agency_title_payroll_agency_code ................ [RUN]
13:51:31  5 of 7 START test not_null_rpt_agency_title_payroll_title_code ................. [RUN]
13:51:31  2 of 7 NO-OP exposure commissioners_dashboard .................................. [NO-OP in 0.00s]
13:51:31  6 of 7 START test not_null_rpt_agency_title_payroll_union_desc ................. [RUN]
13:51:31  3 of 7 PA

In [23]:
!dbt build {ci_flags}

13:51:33  Running with dbt=1.12.6
13:51:33  Registered adapter: duckdb=1.11.0
13:51:34  Found 3 models, 5 data tests, 3 sources, 1 exposure, 500 macros
13:51:34  
13:51:34  Concurrency: 4 threads (target='ci')
13:51:34  
13:51:34  1 of 7 START sql table model ci_local_civil_service.rpt_agency_title_payroll ... [RUN]
13:51:34  1 of 7 OK created sql table model ci_local_civil_service.rpt_agency_title_payroll  [OK in 0.22s]
13:51:34  3 of 7 START test accepted_values_rpt_agency_title_payroll_agency_group__UNIFORMED__EDUCATION__HEALTH_HUMAN_SVCS__DISTRICT_ATTORNEYS__OTHER  [RUN]
13:51:34  4 of 7 START test not_null_rpt_agency_title_payroll_agency_code ................ [RUN]
13:51:34  5 of 7 START test not_null_rpt_agency_title_payroll_title_code ................. [RUN]
13:51:34  2 of 7 NO-OP exposure commissioners_dashboard .................................. [NO-OP in 0.00s]
13:51:34  6 of 7 START test not_null_rpt_agency_title_payroll_union_desc ................. [RUN]
13:51:34  5 of 7 PA

- Everything passes, and only the changed model was rebuilt
- The staging tables came from prod (`--defer`). Here are the tables the CI build actually read:

In [24]:
compiled = Path("target/compiled/civil_service/models/legacy/rpt_agency_title_payroll.sql").read_text()
for table in sorted(set(re.findall(r'"prod"\."\w+"\."\w+"', compiled))):
    print(table)

"prod"."main_civil_service"."stg_civil_servants"
"prod"."main_civil_service"."stg_civil_service_titles"


- Prod and CI tables live side by side, so we can compare them directly:

In [25]:
query("""
    select 'prod' as version, sum(headcount) as headcount, sum(total_annual_salary) as total_annual_salary
    from main_civil_service.rpt_agency_title_payroll
    union all
    select 'ci', sum(headcount), sum(total_annual_salary)
    from ci_local_civil_service.rpt_agency_title_payroll
""", db="data/prod.duckdb")

┌─────────┬───────────┬─────────────────────┐
│ version │ headcount │ total_annual_salary │
│ varchar │  int128   │    decimal(38,2)    │
├─────────┼───────────┼─────────────────────┤
│ prod    │    219894 │      18136814408.02 │
│ ci      │    219894 │      18136814408.02 │
└─────────┴───────────┴─────────────────────┘



- Same headcount and payroll as prod: the new column didn't change any numbers
- Ready to merge

## Clean up

- Put `models/` back the way it is on `main`

In [26]:
!git restore models
!git status --short

 D presentation/demo.ipynb
?? demo.ipynb
